In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ===== Paths (Kaggle) =====
DATA = Path("/kaggle/input/piash-dpfn-mabsa-data")
SENTIC_XLSX = Path("/kaggle/input/piash-senticnet/senticnet.xlsx")

print("DATA:", DATA)
print("SENTIC_XLSX:", SENTIC_XLSX)


DATA: /kaggle/input/piash-dpfn-mabsa-data
SENTIC_XLSX: /kaggle/input/piash-senticnet/senticnet.xlsx


In [2]:
sentic_df = pd.read_excel(SENTIC_XLSX)

SENTIC = {}
for _, r in sentic_df.iterrows():
    c = r.get("CONCEPT", None)
    p = r.get("POLARITY INTENSITY", None)
    if isinstance(c, str):
        c = c.strip().lower()
        if not c:
            continue
        try:
            SENTIC[c] = float(p)
        except:
            pass

print("Sentic concepts loaded:", len(SENTIC))
print("Example:", list(SENTIC.items())[:5])


Sentic concepts loaded: 295292
Example: [('aah', -0.61), ('abacinate', -0.799), ('abacterial', 0.196), ('abaction', -0.329), ('abactor', -0.329)]


In [3]:
def read_tsv(path: Path, split: str):
    df = pd.read_csv(path, sep="\t")
    if split in ("train","dev"):
        df = df.rename({
            "#1 Label": "sentiment",
            "#2 ImageID": "image_id",
            "#3 String": "tweet_content",
            "#3 String.1": "target",
        }, axis=1).drop(["index"], axis=1)
    else:
        df = df.rename({
            "index": "sentiment",
            "#1 ImageID": "image_id",
            "#2 String": "tweet_content",
            "#2 String.1": "target",
        }, axis=1)
    return df

train_df = read_tsv(DATA/"train.tsv", "train")
dev_df   = read_tsv(DATA/"dev.tsv",   "dev")
test_df  = read_tsv(DATA/"test.tsv",  "test")

print("Rows:", len(train_df), len(dev_df), len(test_df))
train_df.head()


Rows: 3179 1122 1037


,sentiment,image_id,tweet_content,target
0,2,1860693.jpg,RT @ ltsChuckBass : $T$ is everything # MCM,Chuck Bass
1,1,1860693.jpg,RT @ ltsChuckBass : Chuck Bass is everything $T$,# MCM
2,1,195984.jpg,$T$ September 7 - - - - Shut Down New York City,Millions For Trayvon
3,0,195984.jpg,Millions For Trayvon September 7 - - - - Shut ...,New York City
4,2,871618.jpg,Me as $T$ .,John Lennon


In [4]:
def ke_gcn_words(tweet_with_T: str, aspect: str):
    # MUST match original adjacency nodes exactly (whitespace split)
    left, right = [s.strip() for s in str(tweet_with_T).split("$T$")]
    return (left + " " + str(aspect) + " " + right).split()

def norm_tok(w: str) -> str:
    # normalization ONLY for lookup, does NOT change nodes
    w = str(w).lower()
    w = re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", w)
    return w

def sentic_score(tok: str) -> float:
    t = norm_tok(tok)
    if not t:
        return 0.0
    return float(SENTIC.get(t, 0.0))


In [5]:
def sentic_coverage_for_df(df: pd.DataFrame, split_name: str):
    """
    Coverage for affective graph word-nodes only.
    - token_coverage: % of word nodes with Sentic polarity != 0
    - tweet_coverage: % of tweets containing >=1 Sentic-hit word
    """
    total_tokens = 0
    hit_tokens = 0

    tweet_has_hit = 0
    hits_per_tweet = []

    hit_vocab = Counter()  # normalized tokens that hit

    for i in range(len(df)):
        tweetT = df.iloc[i]["tweet_content"]
        aspect = df.iloc[i]["target"]

        words = ke_gcn_words(tweetT, aspect)
        scores = [sentic_score(w) for w in words]

        h = sum(1 for s in scores if s != 0.0)
        n = len(words)

        total_tokens += n
        hit_tokens += h

        hits_per_tweet.append(h)
        if h > 0:
            tweet_has_hit += 1
            for w, s in zip(words, scores):
                if s != 0.0:
                    hit_vocab[norm_tok(w)] += 1

    token_cov = (hit_tokens / total_tokens) if total_tokens else 0.0
    tweet_cov = (tweet_has_hit / len(df)) if len(df) else 0.0

    return {
        "split": split_name,
        "tweets": int(len(df)),
        "total_tokens": int(total_tokens),
        "hit_tokens": int(hit_tokens),
        "token_coverage": float(token_cov),
        "tweet_coverage": float(tweet_cov),
        "avg_hits_per_tweet": float(np.mean(hits_per_tweet)) if hits_per_tweet else 0.0,
        "median_hits_per_tweet": float(np.median(hits_per_tweet)) if hits_per_tweet else 0.0,
        "top_hit_tokens": hit_vocab.most_common(20),
    }


In [6]:
train_cov = sentic_coverage_for_df(train_df, "train")
dev_cov   = sentic_coverage_for_df(dev_df,   "dev")
test_cov  = sentic_coverage_for_df(test_df,  "test")

all_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
overall_cov = sentic_coverage_for_df(all_df, "overall")

def cov_row(c):
    return {
        "split": c["split"],
        "tweets": c["tweets"],
        "total_tokens": c["total_tokens"],
        "hit_tokens": c["hit_tokens"],
        "token_coverage_%": round(c["token_coverage"] * 100, 2),
        "tweet_coverage_%": round(c["tweet_coverage"] * 100, 2),
        "avg_hits/tweet": round(c["avg_hits_per_tweet"], 3),
        "median_hits/tweet": round(c["median_hits_per_tweet"], 3),
    }

cov_table = pd.DataFrame([
    cov_row(train_cov),
    cov_row(dev_cov),
    cov_row(test_cov),
    cov_row(overall_cov),
])

cov_table


,split,tweets,total_tokens,hit_tokens,token_coverage_%,tweet_coverage_%,avg_hits/tweet,median_hits/tweet
0,train,3179,53149,11119,20.92,95.22,3.498,3.0
1,dev,1122,18783,3976,21.17,96.17,3.544,3.0
2,test,1037,17677,3767,21.31,95.27,3.633,3.0
3,overall,5338,89609,18862,21.05,95.43,3.534,3.0
